# Analisis de los precios de supermercados en AMBA

Origen de los datos: [SEPA](https://datos.produccion.gob.ar/dataset/sepa-precios)

**Fuentes**:

[2018-2023](https://drive.google.com/drive/folders/13GONeBs5lQCSUdBioHYk-8GhfDtIyliD)

[2024-2026](https://drive.google.com/drive/folders/1GNs9SrZ4BIoBsviBVWYYqRcsj4dwPF-I)

[Últimos meses](https://uadeeduar-my.sharepoint.com/my?id=%2Fpersonal%2Fsriverti%5Fuade%5Fedu%5Far%2FDocuments%2Fbases%5Fsepa&ct=1778176849047&or=Teams%2DHL&ga=1&LOF=1)

In [10]:
# ============================================================
# BLOQUE 1 — Imports, configuración y definición de canasta
# ============================================================
import os, re, gzip, gc
import pandas as pd
import numpy as np

# Rutas
ARCHIVO_1 = "/content/042026_pais_parte1COMPLETO.csv.gz"
ARCHIVO_2 = "/content/042026_pais_parte2COMPLETO.csv.gz"
PATH_MAESTRO_PROD = "/content/Maestro de Productos Interno.xlsx"
PATH_MAESTRO_SUC = "/content/maestro_sucursales_completo.xlsx"

# Verificación
for p in [ARCHIVO_1, ARCHIVO_2, PATH_MAESTRO_PROD, PATH_MAESTRO_SUC]:
    if os.path.exists(p):
        size_mb = os.path.getsize(p) / 1024 / 1024
        print(f"✅ {p}  ({size_mb:.1f} MB)")
    else:
        print(f"❌ NO ENCONTRADO: {p}")

# Canasta de 30 productos (igual a la del análisis anterior)
CANASTA = {
    '7790742363008': ('Leche entera 1L',         20, 'Lácteos'),
    '7791337007628': ('Yogur 190g',               8, 'Lácteos'),
    '7791337061361': ('Queso Casancrem 290g',     2, 'Lácteos'),
    '7793940052002': ('Manteca 100g',             2, 'Lácteos'),
    '7791337007253': ('Cindor 1L',                4, 'Lácteos'),
    '7790272001029': ('Aceite girasol 1,5L',      2, 'Almacén'),
    '7790070433114': ('Arroz 500g',               2, 'Almacén'),
    '7790070320285': ('Fideos 500g',              4, 'Almacén'),
    '7792180140708': ('Harina leudante 1kg',      2, 'Almacén'),
    '7792710000182': ('Yerba 500g',               2, 'Almacén'),
    '7790550000157': ('Café 250g',                1, 'Almacén'),
    '7790040143234': ('Chocolinas 250g',          4, 'Almacén'),
    '7790072002080': ('Sal fina 500g',            1, 'Almacén'),
    '7790895000232': ('Coca Cola lata',           8, 'Bebidas'),
    '7790895067570': ('Coca Sin Azúcar 2,25L',    4, 'Bebidas'),
    '7798062548716': ('Agua Levite 500ml',        8, 'Bebidas'),
    '7793147118860': ('Cerveza lata',             6, 'Bebidas'),
    '7798074864675': ('Vino Malbec 750ml',        2, 'Bebidas'),
    '7790132098459': ('Lavandina 1L',             2, 'Limpieza'),
    '7791290794054': ('Detergente 300ml',         2, 'Limpieza'),
    '7793253003500': ('Limpiador Poett 900ml',    2, 'Limpieza'),
    '7791293047447': ('Shampoo 400ml',            1, 'Higiene'),
    '7791293045948': ('Acondicionador 340ml',     1, 'Higiene'),
    '7791293051208': ('Jabón tocador 90g',        4, 'Higiene'),
    '7791293049557': ('Antitranspirante',         2, 'Higiene'),
    '7891024183083': ('Hilo dental',              1, 'Higiene'),
    '7790770601899': ('Toallas femeninas x16',    2, 'Higiene'),
    '7790250015840': ('Papel higiénico',          2, 'Higiene'),
    '7790580327415': ('Rocklets 40g',             2, 'Snacks'),
    '7790580716707': ('Saladix 100g',             2, 'Snacks'),
}

CANASTA_EANS_LSTRIP = {e.lstrip('0') for e in CANASTA.keys()}
print(f"\n✅ Canasta definida: {len(CANASTA)} productos")

✅ /content/042026_pais_parte1COMPLETO.csv.gz  (147.9 MB)
✅ /content/042026_pais_parte2COMPLETO.csv.gz  (152.4 MB)
✅ /content/Maestro de Productos Interno.xlsx  (20.3 MB)
✅ /content/maestro_sucursales_completo.xlsx  (0.5 MB)

✅ Canasta definida: 30 productos


In [11]:
# ============================================================
# BLOQUE 2 — Cargar maestros y filtrar sucursales del área de interés
# ============================================================
print("Cargando maestros...")
maestro_prod = pd.read_excel(PATH_MAESTRO_PROD)
maestro_suc = pd.read_excel(PATH_MAESTRO_SUC)

print(f"  Productos:  {len(maestro_prod):,} filas")
print(f"  Sucursales: {len(maestro_suc):,} filas")

# Mostrar columnas disponibles del maestro de sucursales para verificación
print(f"\nColumnas del maestro de sucursales:")
print(list(maestro_suc.columns))

# Detectar el campo de provincia (puede llamarse 'PROVINCIA' o 'sucursales_provincia')
posibles_cols_prov = ['PROVINCIA', 'sucursales_provincia', 'provincia']
COL_PROV = next((c for c in posibles_cols_prov if c in maestro_suc.columns), None)
print(f"\n📍 Columna de provincia detectada: '{COL_PROV}'")

# Mostrar valores únicos de provincia (para confirmar nombres exactos)
print(f"\nValores únicos en '{COL_PROV}':")
print(maestro_suc[COL_PROV].value_counts().head(30))

Cargando maestros...
  Productos:  176,702 filas
  Sucursales: 3,611 filas

Columnas del maestro de sucursales:
['id_comercio', 'id_bandera', 'id_sucursal', 'sucursales_nombre', 'sucursales_tipo', 'sucursales_calle', 'sucursales_numero', 'sucursales_latitud', 'sucursales_longitud', 'sucursales_observaciones', 'sucursales_barrio', 'sucursales_codigo_postal', 'sucursales_localidad', 'sucursales_provincia', 'sucursales_lunes_horario_atencion', 'sucursales_martes_horario_atencion', 'sucursales_miercoles_horario_atencion', 'sucursales_jueves_horario_atencion', 'sucursales_viernes_horario_atencion', 'sucursales_sabado_horario_atencion', 'sucursales_domingo_horario_atencion', 'PROVINCIA', 'REGION']

📍 Columna de provincia detectada: 'PROVINCIA'

Valores únicos en 'PROVINCIA':
PROVINCIA
Provincia de Buenos Aires          1295
Ciudad Autónoma de Buenos Aires    1100
Córdoba                             190
Entre Ríos                          134
Santa Fe                            132
Río Negro 